<a href="https://colab.research.google.com/github/SergiSama/UIC-CRB1-2026-2027/blob/main/m1_matplotlib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 · Notebook 4 — Matplotlib for Bioengineers
### Computing, Robotics & Bionics · companion to A. Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*

A **self-contained** plotting reference — no other notebook needed. It covers everything in Géron's
`tools_matplotlib`, worked on biomedical data: a synthetic ECG, the breast-cancer features, and the
digits images. The `imshow` section deliberately sets up the **image-processing module (Module 4)**.

Run in **Google Colab** (*Runtime → Run all*). Everything is offline.

**Contents**
1. The figure/axes model
2. Line plots (an ECG)
3. Scatter plots
4. Histograms and bar charts
5. Subplots — a grid of panels
6. Styling, annotation, saving
7. `imshow` — images and heatmaps (sets up Module 4)
8. Plotting straight from Pandas

Most sections end with an **Exercise**; run the **Solution** cell to check.

---
*Attribution: adapted from Aurélien Géron's `tools_matplotlib.ipynb` ([github.com/ageron/handson-mlp](https://github.com/ageron/handson-mlp)), © Aurélien Géron, licensed under the [Apache License 2.0](https://github.com/ageron/handson-mlp/blob/main/LICENSE). Modified: reordered, extended with biomedical examples and exercises, and adapted for this course.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
print("matplotlib ready")

---
## 1 · The figure/axes model

A **figure** is the whole canvas; it holds one or more **axes** (the actual plots). The explicit style
`fig, ax = plt.subplots()` then `ax.plot(...)` is what Géron uses and scales cleanly to multi-panel
figures. Call `plt.show()` to render.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([0, 1, 2, 3], [0, 1, 4, 9], marker="o")
ax.set_xlabel("x"); ax.set_ylabel("y = x^2"); ax.set_title("A figure with one axes")
plt.show()

---
## 2 · Line plots (an ECG)

Line plots are the natural view of a time signal. We rebuild the synthetic ECG from the NumPy notebook
and plot it.

In [ ]:
def gaussian(t, c, w, a): return a * np.exp(-0.5 * ((t - c) / w) ** 2)
fs, dur, hr = 250, 6.0, 72
t = np.arange(0, dur, 1/fs); beat = 60/hr
ecg = np.zeros_like(t)
for k in range(int(dur/beat) + 1):
    c = k*beat
    ecg += gaussian(t, c-0.12, 0.025, 0.12) + gaussian(t, c, 0.012, 1.0)
    ecg += gaussian(t, c-0.02, 0.012, -0.18) + gaussian(t, c+0.02, 0.012, -0.25)
    ecg += gaussian(t, c+0.18, 0.040, 0.30)
ecg += rng.normal(0, 0.02, t.shape)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t, ecg, lw=0.9, color="firebrick")
ax.set_xlim(0, 3); ax.set_xlabel("time (s)"); ax.set_ylabel("amplitude (mV)")
ax.set_title("Synthetic ECG"); ax.grid(alpha=0.3)
plt.show()

**Exercise 2.** Plot only the first 1.5 seconds of the ECG, add a horizontal dashed line at the
R-peak detection threshold `y = 0.8`, and label it via a legend.

In [ ]:
# Solution 2
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, ecg, lw=1.0, label="ECG")
ax.axhline(0.8, ls="--", color="gray", label="threshold")
ax.set_xlim(0, 1.5); ax.set_xlabel("time (s)"); ax.set_ylabel("mV"); ax.legend()
plt.show()

---
## 3 · Scatter plots

Scatter plots reveal relationships and class structure. We colour breast-cancer samples by diagnosis to
see how two features separate the classes.

In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)
X = bc.frame; y = bc.target           # 0 = malignant, 1 = benign

fig, ax = plt.subplots(figsize=(6, 5))
for cls, name, col in [(0, "malignant", "crimson"), (1, "benign", "steelblue")]:
    m = (y == cls)
    ax.scatter(X.loc[m, "mean radius"], X.loc[m, "mean texture"],
               s=12, alpha=0.6, color=col, label=name)
ax.set_xlabel("mean radius"); ax.set_ylabel("mean texture")
ax.set_title("Two features separate the classes"); ax.legend()
plt.show()

---
## 4 · Histograms and bar charts

Histograms show a distribution; bar charts compare categories. Overlaying class histograms is a fast way
to judge how discriminative a feature is.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
# histogram of one feature, per class
for cls, name, col in [(0, "malignant", "crimson"), (1, "benign", "steelblue")]:
    axes[0].hist(X.loc[y == cls, "mean concavity"], bins=30, alpha=0.6, color=col, label=name)
axes[0].set_xlabel("mean concavity"); axes[0].set_ylabel("count"); axes[0].legend()
axes[0].set_title("Histogram by class")
# bar chart of class counts
counts = [int((y == 0).sum()), int((y == 1).sum())]
axes[1].bar(["malignant", "benign"], counts, color=["crimson", "steelblue"])
axes[1].set_ylabel("count"); axes[1].set_title("Class balance")
plt.tight_layout(); plt.show()

---
## 5 · Subplots — a grid of panels

`plt.subplots(nrows, ncols)` returns an array of axes you index like any NumPy array. Here: the
distribution of the first four features, each in its own panel.

In [ ]:
feats = list(bc.feature_names[:4])
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
for ax, feat in zip(axes.ravel(), feats):
    for cls, col in [(0, "crimson"), (1, "steelblue")]:
        ax.hist(X.loc[y == cls, feat], bins=25, alpha=0.6, color=col)
    ax.set_title(feat, fontsize=10)
fig.suptitle("First four features, by class")
plt.tight_layout(); plt.show()

---
## 6 · Styling, annotation, saving

Control colours, line styles, limits and ticks; annotate points of interest; and save a publication-ready
figure with `savefig` (raise `dpi` for print quality).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, ecg, lw=1.0, color="navy")
# annotate the first detected R-peak
peak_i = int(np.argmax(ecg[:int(0.6*fs)]))
ax.annotate("R peak", xy=(t[peak_i], ecg[peak_i]), xytext=(t[peak_i]+0.2, 1.0),
            arrowprops=dict(arrowstyle="->", color="black"))
ax.set_xlim(0, 1.2); ax.set_xlabel("time (s)"); ax.set_ylabel("mV"); ax.set_title("Annotated ECG")
fig.savefig("ecg_figure.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved ecg_figure.png")

---
## 7 · `imshow` — images and heatmaps (sets up Module 4)

`imshow` displays a 2-D array as an image. This is exactly how the **image-processing module** will show
photographs and masks. Two essentials: choose a **colormap** (`gray` for intensity images) and add a
**colorbar**. We show a handwritten-digit image, then a feature-correlation heatmap.

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
fig, axes = plt.subplots(1, 5, figsize=(10, 2.4))
for ax, img, lab in zip(axes, digits.images[:5], digits.target[:5]):
    ax.imshow(img, cmap="gray")          # an 8x8 intensity image
    ax.set_title(str(lab)); ax.axis("off")
fig.suptitle("imshow on 8x8 digit images (a grayscale image is just a 2-D array)")
plt.show()

In [ ]:
# A heatmap: correlation among the first 8 features
corr = np.corrcoef(X[bc.feature_names[:8]].to_numpy().T)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(8)); ax.set_yticks(range(8))
ax.set_xticklabels(range(8)); ax.set_yticklabels([f[:12] for f in bc.feature_names[:8]], fontsize=8)
fig.colorbar(im, ax=ax, label="correlation")
ax.set_title("Feature correlation heatmap")
plt.show()

**Exercise 7.** Display the *average* image of all digit-`3` samples (mean over `digits.images`
where `digits.target == 3`) with `imshow` and a colorbar. *Hint:* boolean-mask the image stack, then take
the mean over axis 0.

In [ ]:
# Solution 7
mean3 = digits.images[digits.target == 3].mean(axis=0)
fig, ax = plt.subplots(figsize=(3.2, 3))
im = ax.imshow(mean3, cmap="gray"); fig.colorbar(im, ax=ax)
ax.set_title("average '3'"); ax.axis("off")
plt.show()

---
## 8 · Plotting straight from Pandas

Pandas wraps Matplotlib: `df.plot(...)`, `series.hist()`, `series.value_counts().plot.bar()` produce
quick exploratory figures with labels taken from the column names.

In [ ]:
ax = X["mean radius"].plot.hist(bins=30, alpha=0.7, figsize=(6, 3), title="mean radius (Pandas .plot)")
ax.set_xlabel("mean radius")
plt.show()
bc.frame["target"].map({0: "malignant", 1: "benign"}).value_counts().plot.bar(
    color=["crimson", "steelblue"], rot=0, title="class counts (Pandas .plot)")
plt.show()

---
### You now know Matplotlib
The figure/axes model, line/scatter/hist/bar plots, multi-panel grids, styling and annotation, `imshow`
for images and heatmaps, and Pandas' plotting shortcuts. **This completes Module 1.** You can now read and
write the scientific-Python code in Géron's book — **next stop: Chapter 2**, the end-to-end project.
Module 2 develops the linear algebra behind the models; the `imshow` skills here resurface in Module 4
(OpenCV).